# AI-Assisted HDL Verification Pipeline (Prototype)
This Jupyter Notebook demonstrates a minimal prototype of an AI-assisted pipeline that analyzes HDL code against a list of textual requirements using the OpenAI API.

The pipeline consists of:
1. Enter free-form requirements  
2. Normalize requirements via an LLM  
3. Load HDL code  
4. Call the OpenAI API for requirement-to-code matching  
5. Produce a traceability table

**Setup** — Import the demo pipeline helpers so we keep the notebook minimal.

In [37]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

from demo_pipeline import (
    load_requirements,
    load_hdl_code,
    normalize_requirements_with_llm,
    analyze_requirements_with_llm,
    build_traceability_table,
)

**Step 1: Capture free-form requirements** — Use rough wording like you’d get from a notes dump.

In [38]:
raw_requirements = [
    "if an error shows up we should drop into safe mode",
    "after reset/power-up the thing should start in safe state",
    "the counter must not roll over or overflow",
]
raw_requirements

['if an error shows up we should drop into safe mode',
 'after reset/power-up the thing should start in safe state',
 'the counter must not roll over or overflow']

**Step 2: Normalize requirements with the LLM** — Convert free-form bullets into structured ID + text.

In [39]:
requirements = normalize_requirements_with_llm(raw_requirements)
requirements

[{'id': 'R1', 'text': 'If an error occurs, the system shall enter safe mode.'},
 {'id': 'R2',
  'text': 'After reset or power-up, the system shall start in a safe state.'},
 {'id': 'R3', 'text': 'The counter shall not roll over or overflow.'}]

**Step 3: Load HDL code** — Bring in the demo watchdog module to check against.

In [40]:
hdl_code = load_hdl_code()
print(hdl_code)

module watchdog (
    input  wire clk,
    input  wire reset_n,
    input  wire error_flag,
    output reg  [1:0] state
);

localparam SAFE_STATE  = 2'b00;
localparam RUN_STATE   = 2'b01;
localparam ERROR_STATE = 2'b10;

always @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin
        state <= SAFE_STATE;
    end else begin
        case (state)
            SAFE_STATE:  if (!error_flag) state <= RUN_STATE;
            RUN_STATE:   if (error_flag)  state <= ERROR_STATE;
            ERROR_STATE: if (!error_flag) state <= SAFE_STATE;
            default:     state <= SAFE_STATE;
        endcase
    end
end

endmodule



**Step 4: Run LLM requirement-to-code analysis** — Ask the model to map each requirement to HDL evidence.

In [41]:
llm_result = analyze_requirements_with_llm(requirements, hdl_code)
llm_result

[{'id': 'R1',
  'status': 'partial',
  'code_snippet': 'RUN_STATE: if (error_flag) state <= ERROR_STATE;',
  'comment': "The code transitions to ERROR_STATE when an error occurs, which indicates entering an error mode, but there is no explicit indication that ERROR_STATE is a 'safe mode'. Thus, partial compliance."},
 {'id': 'R2',
  'status': 'yes',
  'code_snippet': 'if (!reset_n) begin\n    state <= SAFE_STATE;\nend',
  'comment': 'On reset (active low), the state is set to SAFE_STATE, satisfying the requirement to start in a safe state.'},
 {'id': 'R3',
  'status': 'no',
  'code_snippet': 'No counter present in the design.',
  'comment': 'The code does not include any counter, so this requirement is not addressed.'}]

**Step 5: Build traceability table** — Visualize the results in tabular form.

In [42]:
df = pd.DataFrame(llm_result)
df

,id,status,code_snippet,comment
0,R1,partial,RUN_STATE: if (error_flag) state <= ERROR_STATE;,"The code transitions to ERROR_STATE when an error occurs, which indicates entering an error mode, but there is no explicit indication that ERROR_STATE is a 'safe mode'. Thus, partial compliance."
1,R2,yes,if (!reset_n) begin\n state <= SAFE_STATE;\nend,"On reset (active low), the state is set to SAFE_STATE, satisfying the requirement to start in a safe state."
2,R3,no,No counter present in the design.,"The code does not include any counter, so this requirement is not addressed."


### Summary

This prototype demonstrates a simple AI-assisted pipeline that:
- Captures free-form requirements and normalizes them  
- Loads HDL code  
- Uses an LLM to match requirements to HDL implementation  
- Produces a traceability table

This serves as a foundation for an AI-assisted verification & validation pipeline for FPGA modules.